In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import logging

# Ensure logs are visible in your notebook (using force=True just in case)
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
# 🤫 Silence the noisy external libraries and the text extraction parser
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpxthrottlecache").setLevel(logging.WARNING)
logging.getLogger("src.simple_rag.extraction.parser").setLevel(logging.WARNING)


### Vanguard Index Funds

In [ ]:
from src.simple_rag.extraction.filing_retriever import FilingRetriever
from src.simple_rag.utils.fund_utils import print_fund_info
retriever = FilingRetriever(email_identity="luis.alvarez.conde@alumnos.upm.es")
result = retriever.get_ncsr_filings("VOO", vanguard=True, verbose=True) 


funds = []  
funds.extend(result.funds)                          
  
# Run Verifier
warnings = FilingRetriever.verify_ncsr_data(result)
if not warnings:
    print("All parsed data looks perfectly clean! ✅")
else:
    print("Data issues found:")
    for ticker, issues in warnings.items():
        print(f"  {ticker}: {issues}")   
print(f"{len(funds)} funds from {result.filings_processed} filings")
print_fund_info(funds[:1])

### Vanguard World Fund

In [ ]:
from src.simple_rag.extraction.filing_retriever import FilingRetriever
from src.simple_rag.utils.fund_utils import print_fund_info
retriever = FilingRetriever(email_identity="luis.alvarez.conde@alumnos.upm.es")
result = retriever.get_ncsr_filings("MGK", vanguard=True, verbose=True)  


funds.extend(result.funds)                          

# Run Verifier
warnings = FilingRetriever.verify_ncsr_data(result)
if not warnings:
    print("All parsed data looks perfectly clean! ✅")
else:
    print("Data issues found:")
    for ticker, issues in warnings.items():
        print(f"  {ticker}: {issues}")   
print(f"{len(result.funds)} funds from {result.filings_processed} filings")
print_fund_info(result.funds[:2])

### Vanguard Specialized Funds

In [ ]:
from src.simple_rag.extraction.filing_retriever import FilingRetriever
from src.simple_rag.utils.fund_utils import print_fund_info
retriever = FilingRetriever(email_identity="luis.alvarez.conde@alumnos.upm.es")
result = retriever.get_ncsr_filings("VDIGX", vanguard=True, verbose=True)  


funds.extend(result.funds)                          
  
# Run Verifier
warnings = FilingRetriever.verify_ncsr_data(result)
if not warnings:
    print("All parsed data looks perfectly clean! ✅")
else:
    print("Data issues found:")
    for ticker, issues in warnings.items():
        print(f"  {ticker}: {issues}")   
print(f"{len(result.funds)} funds from {result.filings_processed} filings")
print_fund_info(result.funds[:1])

### Vanguard Whitehall Funds

In [ ]:
from src.simple_rag.extraction.filing_retriever import FilingRetriever
from src.simple_rag.utils.fund_utils import print_fund_info
import gc, pickle
from pathlib import Path

# Add a checkpoint to save the vanguard funds extracted
CHECKPOINT_DIR = Path("edgar_cache/vanguard")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / f"vanguard.pkl"
 

retriever = FilingRetriever(email_identity="luis.alvarez.conde@alumnos.upm.es")
result = retriever.get_ncsr_filings("VMGRX", vanguard=True, verbose=True) 
extracted = result.funds

# Run Verifier
warnings = FilingRetriever.verify_ncsr_data(result)
if not warnings:
    print("All parsed data looks perfectly clean! ✅")
else:
    print("Data issues found:")
    for ticker, issues in warnings.items():
        print(f"  {ticker}: {issues}")   
verified = [f for f in result.funds if f.ticker not in warnings]
print(f"{len(verified)} clean funds from {result.filings_processed} filings")

print_fund_info(verified[:1])

funds.extend(extracted) 
with open(checkpoint_path, "wb") as f:
    pickle.dump(funds, f)

print(f"Saved {len(funds)} funds to {checkpoint_path}")

del result, retriever, verified
gc.collect()


### Method to try the N-CSR parser with a specific CIK

In [ ]:
# ── NCSR parsing + series_id verification ──────────────────────────────────
from src.simple_rag.extraction.filing_retriever import FilingRetriever

EMAIL = "luis.alvarez.conde@alumnos.upm.es"          # required by SEC fair-use policy
CIK   = "0000036405"               # Vanguard Group (change to any CIK)

retriever = FilingRetriever(email_identity=EMAIL)

# Parse one filing (max_filings=1 keeps it fast)
result = retriever.get_ncsr_by_cik(
    cik=CIK,
    max_filings=1,
    vanguard=True,   # set False for non-Vanguard filers
    verbose=True,
)

print(f"\nFilings processed : {result.filings_processed}")
print(f"Funds extracted   : {len(result.funds)}")

---
## Chunk & Section Quality Analysis
Inspect section-level text vs chunk-level text per fund, detect duplicates, and identify the ingestion bug.